# Factual recall via activation patching

**llm-scalpel** · *causal-patcher* · Mechanistic interpretability portfolio demo

This notebook runs a **path patching** (activation patching) experiment on `gpt2-small` using [TransformerLens](https://github.com/TransformerLensOrg/TransformerLens) and the `causal_patcher` API. We compare a **clean** factual prompt (Eiffel Tower → Paris) with a **corrupt** prompt (Colosseum → Rome) and ask: *where* in the network does the model store the “locate the city” behavior, and *which attention heads* implement factual recall?

## 1. Environment and imports

We import `HookedTransformer` for model + cache access and `ExperimentRunner` to run **clean** and **corrupt** baselines, then patch activations from the clean run into the corrupt forward pass at chosen layers and token positions.

If you cloned the repo, the snippet below adds the project root to `sys.path` so `causal_patcher` resolves even without an editable install. For a full environment, run `pip install -e ".[demo]"` from the repository root (installs `pandas` and `ipykernel` for the tables below, plus the package in editable mode).

In [ ]:
import sys
from pathlib import Path

# Resolve llm-scalpel repo root (parent of this notebook)
_here = Path.cwd().resolve()
for candidate in (_here, _here.parent, _here.parent.parent):
    if (candidate / "causal_patcher").is_dir():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

%matplotlib inline
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")


## 2. Load the model

We use **`gpt2-small`** (124M parameters, 12 layers, 12 heads) as a standard lens model: large enough to show nontrivial structure, small enough to iterate quickly on a laptop. The run uses GPU when available.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

from transformer_lens import HookedTransformer

model = HookedTransformer.from_pretrained("gpt2-small", device=device)
model.eval();


## 3. Task: matched factual “city of” prompts

We set up a **contrastive pair** of prompts that share the same template — only the landmark (and implied answer city) differ:

| Role | Text | Desired next-token prediction |
|------|------|----------------------------------|
| **Clean** | `The Eiffel Tower is located in the city of` | **Paris** (token for `" Paris"`) |
| **Corrupt** | `The Colosseum is located in the city of` | **Rome** (token for `" Rome"`) |

**Mechanistic point:** the model’s job on the corrupt run is to predict *Rome*; patching clean activations in tells us which residual/attention sites carry *Paris*–compatible information (here, effectively “Eiffel/Paris” facts) that can *override* the Colosseum context when moved into the wrong prompt.

In [ ]:
# Prompts (trailing space matches common GPT-2 BPE for continuation tokens)
CLEAN_PROMPT = "The Eiffel Tower is located in the city of"
CORRUPT_PROMPT = "The Colosseum is located in the city of"

# Answer strings include a leading space — matches how continuation is usually tokenized
CLEAN_ANSWER_STR = " Paris"
CORRUPT_ANSWER_STR = " Rome"

# Single-token (first token) ids for logit-difference metric
CLEAN_ANSWER_TOKENS = model.to_tokens(CLEAN_ANSWER_STR, prepend_bos=False)
CORRUPT_ANSWER_TOKENS = model.to_tokens(CORRUPT_ANSWER_STR, prepend_bos=False)
assert CLEAN_ANSWER_TOKENS.shape[1] == 1, "Use a single token for clean target"
assert CORRUPT_ANSWER_TOKENS.shape[1] == 1, "Use a single token for corrupt target"
clean_tok = int(CLEAN_ANSWER_TOKENS[0, 0].item())
corrupt_tok = int(CORRUPT_ANSWER_TOKENS[0, 0].item())

CLEAN_TOKS = model.to_tokens(CLEAN_PROMPT, prepend_bos=False)
CORRUPT_TOKS = model.to_tokens(CORRUPT_PROMPT, prepend_bos=False)
print("Clean shape:", tuple(CLEAN_TOKS.shape), " Corrupt shape:", tuple(CORRUPT_TOKS.shape))
print(f"Target token ids:  Paris (clean) = {clean_tok!r}   |   Rome (corrupt) = {corrupt_tok!r}")


## 4. Token alignment (clean vs corrupt)

GPT-2 uses byte-pair encoding: the *Eiffel* vs *Colosseum* **subject** spans are not the same BPE pieces, but here both full prompts have **11 tokens** of the **same length**, so identity alignment `(i, i)` is valid. When lengths differ, you would define a custom list of pairs `(clean_index, corrupt_index)` per site.

Below we list **index**, **id**, and **decoded chunk** for each position. The column **aligned pair** gives `(clean_index, corrupt_index)` used in `PatchTarget(pos=...)`. For these prompts both sequences have the **same length**, so we use identity alignment `(i, i)` — still written explicitly to match the path-patching idiom and to generalize to mismatched lengths (custom pairs per position).

In [ ]:
def show_tokens(toks, label):
    return [
        {
            f"{label} index": i,
            "token_id": int(t),
            "decoded": repr(model.tokenizer.decode([t])),
        }
        for i, t in enumerate(toks[0].tolist())
    ]


d_clean = show_tokens(CLEAN_TOKS, "clean")
d_corrupt = show_tokens(CORRUPT_TOKS, "corrupt")
df = pd.DataFrame(
    {
        "clean index": [x["clean index"] for x in d_clean],
        "clean id": [x["token_id"] for x in d_clean],
        "clean text": [x["decoded"] for x in d_clean],
        "corrupt index": [x["corrupt index"] for x in d_corrupt],
        "corrupt id": [x["token_id"] for x in d_corrupt],
        "corrupt text": [x["decoded"] for x in d_corrupt],
    }
)
df["(clean, corrupt)"] = list(zip(df["clean index"], df["corrupt index"]))

n = len(df)
assert all(
    t[0] == t[1] and 0 <= t[0] < n for t in df["(clean, corrupt)"]
), "Identity alignment (i→i) for all positions"
display(df)


## 5. Baseline: logit difference on clean vs corrupt

`ExperimentRunner` **tokenizes** both strings, runs **`run_with_cache`** for each, and implements the metric

\[
\text{logit\_diff} = \ell_{\text{Paris}} - \ell_{\text{Rome}}
\]

at a chosen final sequence position (here: last prompt token, where the next-token distribution is read).

*Interpretation:* On the **clean** run we expect Paris to be favored over Rome (large positive). On the **corrupt** run, Rome is appropriate — the difference is typically *smaller* or negative. Patching is measured on the **corrupt** run after interventions.

In [ ]:
from causal_patcher import ExperimentRunner, viz
from causal_patcher.targets import PatchTarget

# Cache hook activations for all common patch sites (needed for every layer/position sweep)
names_all = ExperimentRunner.all_patch_hook_names(
    model.cfg.n_layers, kinds=("resid_pre", "attn_head_z")
)

runner = ExperimentRunner(
    model,
    clean_prompt=CLEAN_PROMPT,
    corrupt_prompt=CORRUPT_PROMPT,
    clean_answer_token=clean_tok,
    corrupt_answer_token=corrupt_tok,
    run_baselines=True,
    names_filter=names_all,
)
seq_len = int(runner.corrupt_tokens.shape[-1])
last_pos = seq_len - 1  # "of" — where next-token prediction is read

ld_clean = float(runner.logit_diff(runner.clean_logits, seq_pos=-1).detach().cpu())
ld_corrupt = float(runner.logit_diff(runner.corrupt_logits, seq_pos=-1).detach().cpu())
print(f"Baseline logit diff (last position):  clean  = {ld_clean:+.3f}")
print(f"                                         corrupt = {ld_corrupt:+.3f}")


## 6. Residual stream (`resid_pre`): layer–position scan

**Path patching** at `hook_resid_pre` replaces the full residual before layer \(L\) at a chosen *corrupt* position with the value from the *clean* run at a mapped *clean* position. We sweep all layers and all aligned pairs `(clean_idx, corrupt_idx) = (i, i)`.

**What to look for:** If restoring Paris-related information at a **late** position (e.g. *city* / *of*) strongly raises \(\ell_{\text{Paris}} - \ell_{\text{Rome}}\), that site is a candidate **bottleneck** for factual *style* or task framing. If early subject tokens matter, the *entity* is encoded in the residual at those timesteps.

We plot **patched logit diff** (not “effect minus baseline”) — same convention as the library’s heatmap examples; interpret hot cells as *where* a clean activation carries Paris-favoring signal into the corrupt forward pass.

In [ ]:
# Explicit (clean, corrupt) index for each column — here identity, documented for the portfolio
alignment_pairs: list[tuple[int, int]] = [(i, i) for i in range(seq_len)]

n_layers = model.cfg.n_layers
resid_grid = np.zeros((n_layers, len(alignment_pairs)))

for L in range(n_layers):
    for j, (ci, ri) in enumerate(alignment_pairs):
        logits = runner.patch_clean_into_corrupt(
            PatchTarget("resid_pre", L, pos=(ci, ri))
        )
        resid_grid[L, j] = float(runner.logit_diff(logits, seq_pos=-1).detach().cpu())

fig, ax, im = viz.plot_heatmap(
    resid_grid,
    xlabel="Token position (corrupt index; patch source = same clean index)",
    ylabel="Layer (resid_pre)",
    title="Patched logit diff (Paris ⊖ Rome) · resid_pre path patching",
    figsize=(11, 4.2),
    cmap="RdBu_r",
)
# Label x-axis with corrupt subword text (short)
labels = [model.tokenizer.decode([t]) for t in runner.corrupt_tokens[0].cpu().tolist()]
ax.set_xticks(np.arange(len(labels)))
ax.set_xticklabels([repr(s) for s in labels], rotation=45, ha="right", fontsize=8)
fig.suptitle("Eiffel/Paris (clean) activations → Colosseum/Rome (corrupt) run", y=1.02, fontsize=10)
plt.show()


## 7. Attention heads (`attn_head_z`): recalling facts at the final position

`hook_z` holds **per-head** attention outputs (pre–output projection) with shape
`(batch, position, n_heads, d_head)`. Patching a **single head** at the **last prompt token** injects the clean head’s contribution at the position where the model is about to predict the next city token.

**Interpretation:** Heads with strong positive effect (patch raises Paris vs Rome) behave like **“factual recall” or “copy the retrieved entity”** channels on this task; off-diagonal or late-layer heads are common in case studies. Compare against the `resid_pre` map: sometimes heads explain variance that the residual view alone underweights.

In [ ]:
# Final position: (clean, corrupt) = (last_pos, last_pos) — the next-token readout site
head_pos: tuple[int, int] = (last_pos, last_pos)
print(
    f"Attention patch site (last prompt token): pos={head_pos!r}  (decoded: {model.tokenizer.decode([runner.corrupt_tokens[0, -1].item()])!r})"
)

fig2, ax2, im2, z_grid = viz.plot_layer_head_patching(
    runner,
    positions=head_pos,
    title="Patched logit diff (Paris - Rome) · attention z at final token",
    figsize=(9, 4.2),
)
fig2.suptitle("Single-head z patch at the last prompt position", y=1.03, fontsize=11)
fig2.tight_layout()
plt.show()


## 8. What this demo establishes

* **Constrained counterfactuals** (clean vs corrupt prompts) isolate *which* representations move the next-token distribution toward the clean answer.
* **Residual** patches (`resid_pre`) show *where* in depth and *when* in sequence the model encodes information that, when transplanted, restores Paris-leaning behavior.
* **Per-head** patches at the **last token** screen for **factual recall–style** attention heads: candidates for follow-up (attention visualization, DLA, or ablation).

`causal-patcher` keeps hook naming aligned with `transformer_lens.utilities.get_act_name` and makes **explicit** `(clean_index, corrupt_index)` pairs first-class, which is what you need when tokenizers split subjects differently or when intervening on non-parallel time steps.

— *End of demo*